In [1]:
import pandas as pd

In [2]:
fear_greed = pd.read_csv("fear_greed_index.csv")
historical = pd.read_csv("historical_data.csv")

In [3]:
fear_greed.head(), historical.head()


(    timestamp  value classification        date
 0  1517463000     30           Fear  2018-02-01
 1  1517549400     15   Extreme Fear  2018-02-02
 2  1517635800     40           Fear  2018-02-03
 3  1517722200     24   Extreme Fear  2018-02-04
 4  1517808600     11   Extreme Fear  2018-02-05,
                                       Account  Coin  Execution Price  \
 0  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9769   
 1  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9800   
 2  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9855   
 3  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9874   
 4  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9894   
 
    Size Tokens  Size USD Side     Timestamp IST  Start Position Direction  \
 0       986.87   7872.16  BUY  02-12-2024 22:50        0.000000       Buy   
 1        16.00    127.68  BUY  02-12-2024 22:50      986.524596       Buy   
 2       144.09   1150.63  BUY 

In [4]:
print(fear_greed.shape)
print(historical.shape)


(2644, 4)
(211224, 16)


In [5]:
fear_greed.columns

Index(['timestamp', 'value', 'classification', 'date'], dtype='object')

In [6]:
historical.columns

Index(['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side',
       'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL',
       'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID',
       'Timestamp'],
      dtype='object')

In [7]:
# Standardize column names for consistency
fear_greed.rename(columns={
    'classification': 'Classification',
    'date': 'Date'
}, inplace=True)

historical.rename(columns={
    'Timestamp': 'timestamp_unix',
    'Timestamp IST': 'time',
    'Size Tokens': 'size',
    'Closed PnL': 'closedPnL',
    'Start Position': 'start_position',
    'Execution Price': 'execution_price',
    'Side': 'side'
}, inplace=True)

In [8]:
# Parse dates - fear_greed uses YYYY-MM-DD format
fear_greed['Date'] = pd.to_datetime(fear_greed['Date'])

# Parse historical timestamps - Timestamp IST uses DD-MM-YYYY HH:MM format
historical['time'] = pd.to_datetime(historical['time'], format='%d-%m-%Y %H:%M', errors='coerce')

In [9]:
# Extract date part for merging
fear_greed['Date'] = fear_greed['Date'].dt.date
historical['Date'] = historical['time'].dt.date

# Drop rows where date parsing failed
historical = historical.dropna(subset=['Date'])

In [10]:
# Merge on Date - inner join to keep only matching dates
df = pd.merge(historical, fear_greed, on='Date', how='inner')

# Display merge results
print(f"Merged dataset shape: {df.shape}")
print(f"Date range in merged data: {df['Date'].min()} to {df['Date'].max()}")
print(f"Sentiment distribution:\n{df['Classification'].value_counts()}")

Merged dataset shape: (211218, 20)
Date range in merged data: 2023-05-01 to 2025-05-01
Sentiment distribution:
Classification
Fear             61837
Greed            50303
Extreme Greed    39992
Neutral          37686
Extreme Fear     21400
Name: count, dtype: int64


In [11]:
df.head()
df.shape

(211218, 20)

In [12]:
# Rename columns for consistency
df.rename(columns={
    'Size Tokens': 'size',
    'Size USD': 'size_usd',
    'Closed PnL': 'closedPnL'
}, inplace=True)

In [13]:
df['trade_volume'] = df['size'].abs()
df['profit_flag'] = df['closedPnL'] > 0

# Risk proxy using USD exposure
df['risk_score'] = df['size_usd'].abs()

In [14]:
df[['trade_volume', 'profit_flag', 'risk_score']].head()

,trade_volume,profit_flag,risk_score
0,986.87,False,7872.16
1,16.00,False,127.68
2,144.09,False,1150.63
3,142.98,False,1142.04
4,8.73,False,69.75


In [15]:
# Map sentiment - handle all classification types
df['sentiment'] = df['Classification'].map({
    'Fear': 0,
    'Extreme Fear': 0,  # Map extreme fear to 0
    'Greed': 1,
    'Extreme Greed': 1  # Map extreme greed to 1
})

# Drop rows where sentiment mapping failed (if any)
df = df.dropna(subset=['sentiment'])

print(f"Final dataset shape after sentiment mapping: {df.shape}")
print(f"Sentiment distribution (0=Fear, 1=Greed):\n{df['sentiment'].value_counts()}")

Final dataset shape after sentiment mapping: (173532, 24)
Sentiment distribution (0=Fear, 1=Greed):
sentiment
1.0    90295
0.0    83237
Name: count, dtype: int64


In [16]:
df.to_csv("csv_files/processed_data.csv", index=False)